# 08. Spatial Analysis

This notebook studies spatial patterns of groundwater levels using station coordinates.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

## Load Feature-Engineered Dataset

In [ ]:
candidate_paths = [
    Path("../datasets/groundwater_feature_engineered.csv"),
    Path("datasets/groundwater_feature_engineered.csv"),
    Path("groundwater_feature_engineered.csv")
]

dataset_path = next((p for p in candidate_paths if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("groundwater_feature_engineered.csv not found in expected paths.")

df = pd.read_csv(dataset_path, parse_dates=["Data Acquisition Time"])

print(f"Dataset path: {dataset_path.resolve()}")
print("Shape:", df.shape)
print("Stations:", df["Station"].nunique())
df.head()

## Station-Level Summary

For each station, we calculate average groundwater level and variability.

In [ ]:
station_summary = (
    df.groupby("Station", as_index=False)
      .agg({
          "Latitude": "first",
          "Longitude": "first",
          "Groundwater Level Telemetry 6 Hourly (meter)": ["mean", "std", "min", "max", "count"]
      })
)

station_summary.columns = [
    "Station", "Latitude", "Longitude",
    "Avg_GW_Level", "Std_GW_Level", "Min_GW_Level", "Max_GW_Level", "Observations"
]

station_summary = station_summary.sort_values("Avg_GW_Level", ascending=False).reset_index(drop=True)
station_summary.head(10)

## Station Locations (Scatter Plot)

Color shows average groundwater level. Bigger points mean higher variability.

In [ ]:
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    station_summary["Longitude"],
    station_summary["Latitude"],
    c=station_summary["Avg_GW_Level"],
    s=station_summary["Std_GW_Level"].fillna(0).clip(lower=0).values * 70 + 50,
    cmap="viridis",
    alpha=0.85,
    edgecolor="black",
    linewidth=0.4
)

plt.colorbar(scatter, label="Average Groundwater Level (m)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Spatial Distribution of Station-Level Groundwater Levels")
plt.grid(alpha=0.3)
plt.show()

## Highest and Lowest Average Groundwater Stations

In [ ]:
print("Top 5 stations (highest average groundwater level):")
display(station_summary[["Station", "Avg_GW_Level", "Std_GW_Level", "Observations"]].head(5))

print("Bottom 5 stations (lowest average groundwater level):")
display(station_summary[["Station", "Avg_GW_Level", "Std_GW_Level", "Observations"]].tail(5))

## Latitude and Longitude Band Analysis

We split stations into simple north/south and west/east bands using median latitude and longitude.

In [ ]:
lat_median = station_summary["Latitude"].median()
lon_median = station_summary["Longitude"].median()

station_summary["Lat_Band"] = np.where(station_summary["Latitude"] >= lat_median, "North", "South")
station_summary["Lon_Band"] = np.where(station_summary["Longitude"] >= lon_median, "East", "West")
station_summary["Region"] = station_summary["Lat_Band"] + "-" + station_summary["Lon_Band"]

region_summary = station_summary.groupby("Region", as_index=False).agg(
    Avg_GW_Level=("Avg_GW_Level", "mean"),
    Avg_Variability=("Std_GW_Level", "mean"),
    Stations=("Station", "count")
).sort_values("Avg_GW_Level", ascending=False)

region_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=region_summary, x="Region", y="Avg_GW_Level", palette="Blues_d", ax=axes[0])
axes[0].set_title("Average Groundwater Level by Region")
axes[0].set_ylabel("Avg Groundwater Level (m)")

sns.barplot(data=region_summary, x="Region", y="Avg_Variability", palette="Oranges_d", ax=axes[1])
axes[1].set_title("Average Variability by Region")
axes[1].set_ylabel("Average Std Dev")

for ax in axes:
    ax.set_xlabel("Region")

plt.tight_layout()
plt.show()

## Link to Prediction Task

Spatial differences show that stations behave differently in both level and variability.
This supports using location features (Latitude, Longitude, Station_ID) in the ML models to improve prediction quality.